# Multi Phase Training
This notebook demonstrates training in phases, where each phase is persisted as a checkpoint.

## A. Setup

### Import configurations

In [ ]:
from src.config import ModelConfig, TokenizerConfig, TrainingConfig 

TRAINING_EPOCHS = 5 # will be used for all phases so training will proceed 5 epochs at a time (per checkpoint)

model_config = ModelConfig()
tokenizer_config = TokenizerConfig()                                                                                   
training_config = TrainingConfig(epochs=TRAINING_EPOCHS)

### Initialize untrained model

In [ ]:
from src.model.model import NanoLLM

nano_model = NanoLLM(model_config)

### Load Data
This section uses extracted python modules, but it does not use the more automated training runner that orchestrates both data loading and training.

In [ ]:
from src.data.loader import load_text_from_file,calculate_batches, preprocess_data
from src.paths import DEFAULT_DATA_FILE

stories = load_text_from_file(
    file_path=DEFAULT_DATA_FILE, 
    delimiter=tokenizer_config.delimiter,
    max_paragraphs=training_config.max_stories,
)

batches_per_epoch = calculate_batches(len(stories), training_config.batch_size)

dataloader = preprocess_data(                                                                  
    text_blocks=stories,                                                                                                      
    model_config=model_config,
    tokenizer_config=tokenizer_config,
    training_config=training_config,                                                                                 
)

## B. Train model with 5 epoch
In this section, training and checkpoint persistence are managed individually. Alternatively, `runner.py` is available to orchestrate data loading, preprocessing, training, and checkpoint management with a single method call.

In [ ]:
from src.checkpoint import default_checkpoint_path, save_checkpoint, CheckpointMetadata
from src.training.trainer import Trainer
import dataclasses


# RUN TRAINING
# This will take a while depending on number of epochs specified.
trainer1 = Trainer(
    model=nano_model,
    dataloader=dataloader,
    batches_per_epoch=batches_per_epoch,
    training_config=training_config,            # 5-epoch config
)
history1 = trainer1.train()
print(history1)

# PERSIST CHECKPOINT
checkpoint_destination1 = default_checkpoint_path()
metadata1 = CheckpointMetadata(
    cumulative_epochs_completed=training_config.epochs,
    final_loss=history1["train_loss"][-1] if history1["train_loss"] else None,
    model_config=dataclasses.asdict(model_config),
    training_config=dataclasses.asdict(training_config),
    tokenizer_config=dataclasses.asdict(tokenizer_config),
)
save_checkpoint(nano_model, checkpoint_destination1, metadata=metadata1)

In [ ]:
# Confirm that checkpoint metadata correctly reflects actual epochs trained (5)
from src.checkpoint import load_metadata

metadata1 = load_metadata(checkpoint_destination1)
epochs1 = metadata1.cumulative_epochs_completed

print(f"Checkpoint 1 cumulative_epochs_completed: {epochs1} (expected: {TRAINING_EPOCHS})")
assert epochs1  == TRAINING_EPOCHS 

In [ ]:
from src.compare import state_snapshot

# Capture deep copy of state after first round of training
initial_state = state_snapshot(nano_model)

## C. Reload from checkpoint

In [ ]:
import logging
from src.checkpoint import apply_checkpoint

# Create skeleton for the model
reloaded_model = NanoLLM(model_config)

# Reload weights using persisted checkpoint
#    logging lines here mute INFO-level logs (lots of orbax/checkpoint noise) while leaving exceptions and actual errors visible
logging.disable(logging.INFO)                                                                                                                                                                                                                    
apply_checkpoint(reloaded_model, checkpoint_destination1)
logging.disable(logging.NOTSET)                    

## D. Train model with 5 more epochs (total of 10)

In [ ]:
# RUN TRAINING
trainer2 = Trainer(
    model=reloaded_model,
    dataloader=dataloader,
    batches_per_epoch=batches_per_epoch,
    training_config=training_config,            # same 5-epoch config
)
history2 = trainer2.train()
print(history2)

# Cumulative epochs = prior (loaded from checkpoint_path_1) + this run's epochs.
cumulative_epochs = epochs1 + training_config.epochs

# PERSIST CHECKPOINT
checkpoint_destination2 = default_checkpoint_path()
metadata2 = CheckpointMetadata(
    cumulative_epochs_completed=cumulative_epochs,
    final_loss=history2["train_loss"][-1] if history2["train_loss"] else None,
    model_config=dataclasses.asdict(model_config),
    training_config=dataclasses.asdict(training_config),
    tokenizer_config=dataclasses.asdict(tokenizer_config),
)
save_checkpoint(reloaded_model, checkpoint_destination2, metadata=metadata2)

In [ ]:
# Confirm that checkpoint metadata correctly reflects actual epochs trained (5 + 5 = 10)

from src.checkpoint import load_metadata

metadata2 = load_metadata(checkpoint_destination2)  
epochs2 = metadata2.cumulative_epochs_completed

print(f"Checkpoint 2 cumulative_epochs_completed: {epochs2} (expected: {TRAINING_EPOCHS * 2})")
assert epochs2 == TRAINING_EPOCHS * 2

In [ ]:
# Capture deep copy of state after second round of training
resulting_state = state_snapshot(reloaded_model)

## E. Compare the model at both checkpoints
This section uses the extracted python modules. There is also a CLI script available (`uv run nanollm-compare`) which can take optional `--before` and `--after` flag arguments.

In [ ]:
from src.compare import (
    compare_norms,
    compare_states,
    format_norms_comparison,
    format_state_comparison,
)

In [ ]:
# initial_state and resulting_state are already saved to deep copies
# so we don't need to worry about state variables getting mutated by subsequent training
norms_report = compare_norms(initial_state, resulting_state)
width = 50
print(format_norms_comparison(norms_report, width))

In [ ]:
states_report = compare_states(initial_state, resulting_state)
width = 50
print(format_state_comparison(states_report, width))

## F. Alternative Training Flows 

### 1. Use training runner to resume training

In [ ]:
from src.checkpoint import get_latest_checkpoint, build_model_from_checkpoint
from src.training.resume_context import ResumeContext
from src.training.runner import run

# Use extracted code to load a model with pre-trained weights from most recent checkpoint
updated_model, _, tokenizer_config = build_model_from_checkpoint(checkpoint_destination2)

# Run training (includes checkpoint loading and persistence)
resume_ctx = ResumeContext.from_checkpoint(checkpoint_destination2)
checkpoint_destination3 = default_checkpoint_path()
run(
    model=updated_model,
    tokenizer_config=tokenizer_config,
    data_source=DEFAULT_DATA_FILE,
    training_config=training_config,                 # same 5-epoch config
    checkpoint_destination=checkpoint_destination3,
    resume_from=resume_ctx,                          # checkpoint source + prior epochs accumulated
)

In [ ]:
# Confirm that checkpoint metadata correctly reflects actual epochs trained (5 + 5 + 5 = 15)

metadata3 = load_metadata(checkpoint_destination3)  
epochs3 = metadata3.cumulative_epochs_completed
print(f"Checkpoint 3 cumulative_epochs_completed: {epochs3} (expected: {TRAINING_EPOCHS * 3})")
assert epochs3 == TRAINING_EPOCHS * 3

### 2. Use CLI script to resume training
This script can be run without specifying any arguments. When no overrides are passed form the terminal as flag arguments, the script falls back to defaults such as loading the most recent checkpoint as the starting point for the training.

In [ ]:
# shell magic allows us to run the CLI script directly from the notebook
# Use --epochs override to specify the number of subsequent training epochs.
#     cumulative_epochs_completed after this run should be (5*3) + 2 = 17
!uv run nanollm-resume --epochs 2